# Diagnóstico: ¿puede aprender cuatro imágenes?
Aprobado por Fernanda tras el ensayo `nox-text-20261103-104507` fallido.
**Preparado, sin ejecución GPU de este diagnóstico todavía.**

La pregunta es si el proceso puede memorizar tres imágenes positivas y una negativa
que ya pertenecen a train. No se modifica ninguna caja aprobada. No se utiliza val
ni test para elegir ejemplos o ajustar parámetros. Acertar estas mismas imágenes
no demuestra generalización ni completa el esencial.

## Preparación
Usa una sesión nueva de Colab, GPU gratuita. Ejecuta en orden y sube el mismo
`nox-text-dataset-2026-11-03.zip` de 23 imágenes cuando se solicite. El ZIP y los
pesos previos no se sobrescriben. Los permisos de reutilización conservan el estado
pendiente del registro de fuentes. Se mantienen los controles de GPU y de integridad.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.metadata, io, json, math, platform, shutil, subprocess, sys, time, zipfile
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU CUDA. Selecciona GPU gratuita y vuelve a comenzar; no se inicia entrenamiento CPU.")
print("GPU:", torch.cuda.get_device_name(0), "| Python:", sys.version.split()[0], "| Torch:", torch.__version__)
probe = torch.ones((16,16), device="cuda")
assert (probe @ probe)[0,0].item() == 16
del probe
torch.cuda.empty_cache()
WORK = Path("/content/nox-learning-diagnostic")
WORK.mkdir(exist_ok=True)
environment = {"python":sys.version,"platform":platform.platform(),"torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0)}


In [ ]:
import importlib.util
if importlib.util.find_spec("jedi") is None:
    subprocess.run([sys.executable,"-m","pip","install","jedi==0.19.2"],check=True)
subprocess.run([sys.executable,"-m","pip","install","ultralytics==8.4.189"],check=True)
subprocess.run([sys.executable,"-m","pip","check"],check=True)
from ultralytics import YOLO, settings
settings.update({k:False for k in ("sync","wandb","mlflow","comet","clearml") if k in settings})
environment["ultralytics"] = importlib.metadata.version("ultralytics")
freeze = subprocess.run([sys.executable,"-m","pip","freeze"],capture_output=True,text=True,check=True).stdout


In [ ]:
from google.colab import files
from PIL import Image, ImageDraw
import yaml
EXPECTED_ZIP = "8332c1b07824f7c5a2abac9c43f17c23cc78dfa2d7c92d9953af77b4a9364254"
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Selecciona exactamente un ZIP")
filename, content = next(iter(uploaded.items()))
if hashlib.sha256(content).hexdigest() != EXPECTED_ZIP:
    raise ValueError("El ZIP no coincide con el lote aprobado; no se extrae")
DATA = WORK / "dataset"
if DATA.exists():
    raise FileExistsError("El dataset ya está extraído; continúa con la celda siguiente")
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    entries = archive.namelist()
    prefix = "nox-text-2026-11-03/"
    manifest = json.loads(archive.read(prefix+"manifest.json"))
    expected = {prefix+"manifest.json",prefix+"dataset.yaml"}
    expected.update(prefix+i[k] for i in manifest["images"] for k in ("image","label"))
    if len(entries)!=len(set(entries)) or set(entries)!=expected:
        raise ValueError("Contenido de ZIP inesperado")
    if sum(i.file_size for i in archive.infolist()) > 200_000_000:
        raise ValueError("Tamaño de ZIP inesperado")
    for name in entries:
        target = (DATA / name.removeprefix(prefix)).resolve()
        if not target.is_relative_to(DATA.resolve()):
            raise ValueError("Ruta no válida en ZIP")
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(archive.read(name))
for item in manifest["images"]:
    image_path, label_path = DATA/item["image"], DATA/item["label"]
    assert hashlib.sha256(image_path.read_bytes()).hexdigest()==item["source_sha256"]
    assert hashlib.sha256(label_path.read_bytes()).hexdigest()==item["label_sha256"]
    with Image.open(image_path) as image:
        image.load()
        assert image.size==(item["width"],item["height"])
    rows=label_path.read_text().splitlines()
    assert len(rows)==len(item["boxes"])
    for row, original in zip(rows,item["boxes"],strict=True):
        label,*values=row.split()
        assert label=="0" and len(values)==4
        cx,cy,w,h=map(float,values)
        assert all(math.isfinite(v) for v in (cx,cy,w,h)) and w>0 and h>0
        assert -.000001<=cx-w/2<=cx+w/2<=1.000001 and -.000001<=cy-h/2<=cy+h/2<=1.000001
        restored=[(cx-w/2)*item["width"],(cy-h/2)*item["height"],(cx+w/2)*item["width"],(cy+h/2)*item["height"]]
        assert max(abs(a-b) for a,b in zip(restored,original)) < .001
for key in ("source_sha256","video_id","group_id"):
    parts={s:{i[key] for i in manifest["images"] if i["split"]==s} for s in ("train","val","test")}
    assert not parts["train"]&parts["val"] and not parts["train"]&parts["test"] and not parts["val"]&parts["test"]
assert len(manifest["images"])==23
assert {s:sum(i["split"]==s for i in manifest["images"]) for s in ("train","val","test")}=={"train":19,"val":3,"test":1}
config={"path":str(DATA),"train":"images/train","val":"images/val","test":"images/test","names":{0:"nox_texto"}}
(DATA/"dataset.yaml").write_text(yaml.safe_dump(config))
del uploaded,content
print("23 imágenes verificadas:",manifest["summary"])


## 1. Elegir la muestra y fijar el experimento
Se eligen las tres cajas más grandes disponibles en vídeos distintos de train,
conservando todas las cajas de cada imagen, más el primer negativo de train por
nombre. Selección determinista basada en anotaciones, sin mirar predicciones.

Se muestran las cuatro referencias. El YAML apunta train y val a estas mismas
cuatro imágenes únicamente para esta comprobación de memorización: sus métricas
internas **no son validación independiente**.

80 épocas fijas; lote 2; tamaño 640; semilla 42. Sin transformaciones aleatorias.
Adam a 0,001 y `nbs=2`, de modo que se actualicen pesos en cada lote, sin acumular
hasta el lote nominal 64 del ensayo anterior. Son cambios deliberados para facilitar
aprender la muestra; este ensayo no aísla cuál de ellos explica un eventual cambio.
Se usa un checkpoint oficial fresco y se evalúa `last.pt` tras las 80 épocas.


In [ ]:
import shutil
positives=sorted((i for i in manifest['images'] if i['split']=='train' and i['boxes']),
    key=lambda i:(-max((b[2]-b[0])*(b[3]-b[1]) for b in i['boxes']),i['image']))
selected=[]
seen=set()
for item in positives:
    if item['video_id'] not in seen:
        selected.append(item);seen.add(item['video_id'])
    if len(selected)==3:break
selected+=sorted((i for i in manifest['images'] if i['split']=='train' and not i['boxes']),key=lambda i:i['image'])[:1]
assert len(selected)==4 and sum(bool(i['boxes']) for i in selected)==3
assert all(i['split']=='train' for i in selected)
RUN=WORK/('nox-fit-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
RUN.mkdir(parents=True,exist_ok=False)
SAMPLE=RUN/'sample'
from IPython.display import display
for item in selected:
    for kind in ('image','label'):
        target=SAMPLE/('images' if kind=='image' else 'labels')/Path(item[kind]).name
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(DATA/item[kind],target)
    with Image.open(DATA/item['image']) as source:image=source.convert('RGB')
    draw=ImageDraw.Draw(image)
    for box in item['boxes']:draw.rectangle(tuple(box),outline='lime',width=3)
    print(Path(item['image']).name,'| referencia:',len(item['boxes']))
    display(image)
SAMPLE_YAML=SAMPLE/'dataset.yaml'
SAMPLE_YAML.write_text(yaml.safe_dump(dict(path=str(SAMPLE),train='images',val='images',names={0:'nox_texto'})))
PARAMETERS=dict(epochs=80,patience=0,imgsz=640,batch=2,nbs=2,device=0,seed=42,deterministic=True,
    optimizer='Adam',lr0=0.001,lrf=1.0,warmup_epochs=0.0,weight_decay=0.0,
    workers=0,amp=False,save=True,save_period=20,plots=True,cache=False,val=False,
    mosaic=0.0,mixup=0.0,cutmix=0.0,copy_paste=0.0,fliplr=0.0,flipud=0.0,
    hsv_h=0.0,hsv_s=0.0,hsv_v=0.0,scale=0.0,translate=0.0,degrees=0.0,shear=0.0,perspective=0.0)
record=dict(diagnostic='training_sample_fit_only',status='prepared',parameters=PARAMETERS,
    dataset_zip_sha256=EXPECTED_ZIP,selected=selected,independent_evaluation=False)
(RUN/'experiment.json').write_text(json.dumps(record,indent=2))
(RUN/'environment.json').write_text(json.dumps(environment,indent=2))
(RUN/'requirements-resolved.txt').write_text(freeze)
print('Resultados:',RUN)


## 2. Entrenar la muestra
Este paso sí entrena en la GPU. Puede tardar unos minutos; no se garantiza tiempo.
Si aparece error, descarga los resultados disponibles con la última celda.
Una desconexión puede perder los archivos temporales: descárgalos antes de cerrar.


In [ ]:
if not torch.cuda.is_available():raise RuntimeError('GPU CUDA no disponible')
if (RUN/'train').exists():raise FileExistsError('El ensayo ya comenzó; no sobrescribir')
model=YOLO('yolo11n.pt')
record['pretrained_checkpoint_sha256']=hashlib.sha256(Path(model.ckpt_path).read_bytes()).hexdigest()
record['status']='training'
(RUN/'experiment.json').write_text(json.dumps(record,indent=2))
started=time.perf_counter()
try:
    model.train(data=str(SAMPLE_YAML),project=str(RUN),name='train',exist_ok=False,**PARAMETERS)
    assert (RUN/'train/weights/last.pt').is_file()
    record['status']='training_completed'
except BaseException as error:
    record['status']='training_failed_or_interrupted'
    record['error']=type(error).__name__+': '+str(error)
    raise
finally:
    record['elapsed_training_seconds']=time.perf_counter()-started
    (RUN/'experiment.json').write_text(json.dumps(record,indent=2))


## 3. Comprobar si aprendió esas mismas imágenes
Confianza fija 0,25 e IoU mínimo 0,50 para asociar predicción y referencia una a una.
Se cuentan aciertos, falsas detecciones y logos omitidos. El criterio diagnóstico
estricto exige recuperar todas las cajas y no producir falsos positivos en estas
cuatro imágenes. Un fallo no identifica por sí solo la causa; un acierto solo prueba
capacidad de ajuste sobre esta muestra. No usar el resultado como precisión del producto.

Se guardan vistas y JSON. Revisaremos también las pérdidas y las cajas a escala útil.


In [ ]:
def match_boxes(references,boxes,scores,threshold=0.5):
    def iou(a,b):
        overlap=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
        union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-overlap
        return overlap/union if union>0 else 0.0
    remaining=set(range(len(references)));matched=[]
    for index in sorted(range(len(boxes)),key=lambda i:-scores[i]):
        if not remaining:break
        target=max(remaining,key=lambda j:iou(boxes[index],references[j]))
        if iou(boxes[index],references[target])>=threshold:
            remaining.remove(target);matched.append(dict(prediction=index,reference=target,iou=iou(boxes[index],references[target])))
    return dict(tp=len(matched),fp=len(boxes)-len(matched),fn=len(remaining),matches=matched)

last=RUN/'train/weights/last.pt'
if not last.is_file():raise FileNotFoundError('No existe last.pt; revisar entrenamiento')
fitted=YOLO(str(last))
assert fitted.names=={0:'nox_texto'}
results=[]
for item in selected:
    result=fitted.predict(str(DATA/item['image']),conf=0.25,imgsz=640,device=0,save=False,verbose=False)[0]
    boxes=result.boxes.xyxy.cpu().tolist();scores=result.boxes.conf.cpu().tolist()
    assert all(c==0 for c in result.boxes.cls.cpu().tolist())
    metrics=match_boxes(item['boxes'],boxes,scores)
    results.append(dict(image=item['image'],reference_boxes=item['boxes'],boxes=boxes,scores=scores,**metrics))
    target=RUN/'predictions'/Path(item['image']).name;target.parent.mkdir(exist_ok=True)
    result.save(filename=str(target))
    print(Path(item['image']).name,metrics)
    with Image.open(target) as image:display(image.copy())
totals={key:sum(r[key] for r in results) for key in ('tp','fp','fn')}
summary=dict(checkpoint_sha256=hashlib.sha256(last.read_bytes()).hexdigest(),confidence=0.25,iou=0.5,
    independent_evaluation=False,totals=totals,sample_fit_pass=totals['fp']==0 and totals['fn']==0,
    interpretation='Training sample only; not evidence of generalization',predictions=results)
(RUN/'sample-fit.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
print('AJUSTE DE MUESTRA:',summary['sample_fit_pass'],totals)


## 4. Descargar y compartir
Descarga el ZIP de este diagnóstico y el cuaderno ejecutado (Archivo → Descargar
→ Descargar .ipynb). Mantén el identificador `nox-fit-...` para no confundirlo con
el entrenamiento anterior. El ZIP contiene la muestra local, pesos y resultados:
no publicarlo en GitHub. Esta celda funciona también tras un fallo de entrenamiento.


In [ ]:
from google.colab import files
if "RUN" not in globals() or not RUN.exists():
    raise RuntimeError("Todavía no existe una ejecución que descargar")
archive_path=Path(str(RUN)+"-results.zip")
if archive_path.exists():
    raise FileExistsError("Ya hay un ZIP de esta ejecución; no sobrescribir")
with zipfile.ZipFile(archive_path,"x",zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file():
            archive.write(path,path.relative_to(RUN))
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
print("Descarga:",archive_path.name,"| SHA256:",hashlib.sha256(archive_path.read_bytes()).hexdigest())
files.download(str(archive_path))
